# Bölüm 3 — Ayrılma Aksiyomları

Ayrılma aksiyomları, bir topolojik uzaydaki noktaların ve kapalı kümelerin birbirinden
açık kümeler aracılığıyla ne ölçüde "ayrılabildiğini" ölçer. Her aksiyom bir öncekinden
daha güçlüdür.

## 1. Konu

| Aksiyom | İsim | Koşul |
|---------|------|-------|
| **T0** | Kolmogorov | $\forall x \neq y$: $\exists U \in \tau$ ayıran |
| **T1** | Fréchet | $\forall x \neq y$: iki yönlü açık ayrım |
| **T2** | Hausdorff | $\forall x \neq y$: $\exists U \ni x$, $V \ni y$, $U \cap V = \emptyset$ |
| **T2.5** | Urysohn | disjoint kapalı komşuluklar |
| **T3** | Regüler | T1 + nokta/kapalı ayırma |
| **T3.5** | Tychonoff | T1 + sürekli fonksiyon ayırma |
| **T4** | Normal | T1 + disjoint kapalılar ayırma |

**Sıralama:** T4 $\Rightarrow$ T3.5 $\Rightarrow$ T3 $\Rightarrow$ T2.5 $\Rightarrow$ T2 $\Rightarrow$ T1 $\Rightarrow$ T0

## 2. Teoremler

**Teorem 2.1 (Ayrılma Zinciri).** T4 $\Rightarrow$ T3.5 $\Rightarrow$ T3 $\Rightarrow$ T2.5
$\Rightarrow$ T2 $\Rightarrow$ T1 $\Rightarrow$ T0. Tersi genel olarak doğru değildir.

**Teorem 2.2 (Urysohn Fonksiyon Teoremi).** $X$ normal ise ve $C, D$ disjoint kapalı ise,
$f: X \to [0,1]$ sürekli vardır: $f|_C \equiv 0$, $f|_D \equiv 1$.

**Teorem 2.3 (Tietze Genişleme).** $X$ normal ise her kapalı $A \subseteq X$ üzerinde
$f: A \to [a,b]$ süreklisi $X$'e genişletilebilir.

**Teorem 2.4 (Sonlu T1 $\iff$ Ayrık).**
Sonlu uzayda T1 $\iff$ ayrık topoloji.
*Kanıt:* T1 $\Rightarrow$ her tekil küme kapalı $\Rightarrow$ her küme kapalı (sonlu birleşim)
$\Rightarrow$ her küme açık.

## 3. Algoritmalar

### T0 Karar Prosedürü — $O(|X|^2 \cdot |\tau|)$

```
KontrolT0(X, τ):
    for each pair (x, y) with x ≠ y:
        if not (∃U∈τ: x∈U∧y∉U) and not (∃U∈τ: y∈U∧x∉U):
            return False
    return True
```

T2 için çiftlere disjoint açık komşuluklar bulunması gerekir: $O(|X|^2 \cdot |\tau|^2)$.

## 4. pytop API

| Fonksiyon | İmza | Açıklama |
|-----------|------|----------|
| `is_t0` … `is_t4` | `(space)` | Aksiyom kontrolü → `Result` |
| `is_hausdorff`, `is_regular`, `is_normal` | `(space)` | Alias'lar |
| `separation_chain` | `(space)` | Tüm zincir: `dict[str, Result]` |
| `analyze_separation` | `(space, property='hausdorff')` | Tek aksiyom sorgu |

## 5. Örnekler

In [1]:
from pytop import (
    sierpinski_space, discrete_topology, indiscrete_topology,
    naturals_cofinite, real_line_metric,
    is_t0, is_t1, is_t2, is_t2_5, is_t3, is_t4,
    is_hausdorff, is_regular, is_normal, is_perfectly_normal,
    separation_chain, analyze_separation,
)

### Örnek 5.1 — Sierpiński: T0 ✓, T1 ✗

In [2]:
s = sierpinski_space()
print("T0:", is_t0(s).status)
print("T1:", is_t1(s).status)
print("T2:", is_t2(s).status)

T0: true
T1: false
T2: false


**Çıktı Açıklaması:** $\{1\}$ açık → $0$ ve $1$ T0 ayrımı mümkün. Ancak $\{0\}$ açık
değil → T1 başarısız.

### Örnek 5.2 — İndirgenmiş: Hiçbir T Aksiyomu

In [3]:
ind = indiscrete_topology('a', 'b')
print("T0:", is_t0(ind).status)
print("T1:", is_t1(ind).status)

T0: false
T1: false


**Çıktı Açıklaması:** $\tau = \{\emptyset, X\}$; $a$ ve $b$'yi ayıran açık küme yok.

### Örnek 5.3 — Kosonlu $\mathbb{N}$: T1 ✓, T2 ✗

In [4]:
nc = naturals_cofinite()
print("T0:", is_t0(nc).status)
print("T1:", is_t1(nc).status)
print("T2:", is_t2(nc).status)

T0: true
T1: true
T2: false


**Çıktı Açıklaması:** Kosonlu topolojide her tekil küme kapalıdır → T1. Fakat
herhangi iki boş olmayan açık küme kesişir → T2 başarısız.

### Örnek 5.4 — Ayrılma Zinciri: Ayrık Topoloji

In [5]:
d = discrete_topology(1, 2, 3)
chain = separation_chain(d)
for prop, result in chain.items():
    print(f"  {prop:20s}: {result.status}")

  t0                  : true
  t1                  : true
  hausdorff           : true
  urysohn             : true
  t3                  : true
  tychonoff           : true
  t4                  : true
  completely_normal   : true
  perfectly_normal    : true


**Çıktı Açıklaması:** Ayrık topoloji tüm aksiyomları sağlar.

### Örnek 5.5 — Sierpiński Zinciri

In [6]:
for prop, result in separation_chain(s).items():
    print(f"  {prop:20s}: {result.status}")

  t0                  : true
  t1                  : false
  hausdorff           : false
  urysohn             : false
  t3                  : false
  tychonoff           : unknown
  t4                  : false
  completely_normal   : false
  perfectly_normal    : false


**Çıktı Açıklaması:** Yalnızca T0 sağlanır. `tychonoff: unknown` — sembolik çıkarım sınırı.

### Örnek 5.6 — analyze_separation

In [7]:
rl = real_line_metric()
print("Real line Hausdorff?  :", analyze_separation(rl).status)
print("Discrete Hausdorff?   :", analyze_separation(d).status)
print("Sierpinski Hausdorff? :", analyze_separation(s).status)
print("Sierpinski T0?        :", analyze_separation(s, 't0').status)

Real line Hausdorff?  : true
Discrete Hausdorff?   : true
Sierpinski Hausdorff? : false
Sierpinski T0?        : true


**Çıktı Açıklaması:** `analyze_separation(space, property)` tek aksiyom sorgular.
`status='true'` → uzay o aksiyomu sağlar. Varsayılan `'hausdorff'`'tur.

## 6. Alıştırmalar

**K1.** `make_topology({1,2,3},{1},{2},{1,2})` için `separation_chain` çalıştırın.

**K2.** `finite_chain_space(3)` üzerinde en yüksek sağlanan aksiyom hangisidir?

**K3.** `two_point_indiscrete_space()` üzerinde T0, T1, T2 test edin.

**T1.** T2 $\Rightarrow$ T1 $\Rightarrow$ T0 implicasyonlarını kanıtlayın.

**T2.** Sonlu uzayda T1 $\iff$ ayrık topoloji olduğunu gösterin.